In [1]:
%pip install -q transformers pillow matplotlib pandas requests

In [2]:
import io
import json
import random
import zipfile
from pathlib import Path

import requests
import torch
import torch.nn.functional as F
import transformers
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, CLIPModel

In [3]:
ROOT = Path.cwd()

DATA = ROOT / "data"
IMAGES = DATA / "images"
OUTPUTS = ROOT / "outputs"

for folder in (DATA, IMAGES, OUTPUTS):
    folder.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Folder:", ROOT)
print("Device:", device)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

Folder: /content
Device: cuda
PyTorch: 2.11.0+cu130
Transformers: 5.18.0


In [ ]:
manifest_path = DATA / "xm3600_subset500.json"

if not manifest_path.exists():
    url = "https://google.github.io/crossmodal-3600/web-data/captions.zip"

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    eligible = []

    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        with archive.open("captions.jsonl") as source:
            for line in source:
                item = json.loads(line)

                en = item.get("en", {}).get("caption", [])
                bn = item.get("bn", {}).get("caption", [])

                if en and bn:
                    eligible.append({
                        "image_id": item["image/key"],
                        "en": en,
                        "bn": bn,
                    })

    eligible.sort(key=lambda item: item["image_id"])
    subset = random.Random(42).sample(eligible, 500)

    manifest_path.write_text(
        json.dumps(subset, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

subset = json.loads(manifest_path.read_text(encoding="utf-8"))
print("Fixed image count:", len(subset))

Fixed image count: 500
